In [1]:
import os, gc, joblib
import numpy as np
import tensorflow as tf

X_PATH = "../data/processed/X_all.npy"
C_PATH = "../data/processed/C_all.npy"

X_all = np.load(X_PATH, allow_pickle=False)
C_all = np.load(C_PATH, allow_pickle=False)

print("Loaded ✅")
print("X_all:", X_all.shape, X_all.dtype)
print("C_all:", C_all.shape, C_all.dtype)

if np.isnan(X_all).mean() > 0:
    X_all = np.nan_to_num(X_all, nan=0.0)

os.makedirs("../saved_model", exist_ok=True)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, C_tr, C_val = train_test_split(
    X_all, C_all,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Split ✅")
print("X_tr :", X_tr.shape, "C_tr :", C_tr.shape)
print("X_val:", X_val.shape, "C_val:", C_val.shape)

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler

scaler = RobustScaler(quantile_range=(25, 75))

N_tr, T, F = X_tr.shape
scaler.fit(X_tr.reshape(N_tr*T, F))

def apply_scaler(sc, X):
    N, T, F = X.shape
    X2 = sc.transform(X.reshape(N*T, F))
    return X2.reshape(N, T, F).astype(np.float32)

X_trs  = apply_scaler(scaler, X_tr)
X_vals = apply_scaler(scaler, X_val)

ctx_scaler = StandardScaler()
C_tr  = ctx_scaler.fit_transform(C_tr).astype(np.float32)
C_val = ctx_scaler.transform(C_val).astype(np.float32)

joblib.dump(scaler, "../saved_model/scaler.joblib")
joblib.dump(ctx_scaler, "../saved_model/ctx_scaler.joblib")

WINDOW_SIZE = X_trs.shape[1]
N_FEATURES  = X_trs.shape[2]
TOPK = max(10, int(0.10 * (WINDOW_SIZE * N_FEATURES)))

print("Scaled ✅")
print("TOPK =", TOPK)

In [ ]:

import os, sys
sys.path.append(os.path.abspath(".."))

from src.model import build_film_ae, FiLM  # must exist in src/model.py

In [ ]:
CTX_DIM = C_tr.shape[1]
model = build_film_ae(WINDOW_SIZE, N_FEATURES, CTX_DIM, units=64, latent=64)
model.summary()

In [ ]:
cb = [
    tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5, min_lr=1e-6),
]

history = model.fit(
    [X_trs, C_tr], X_trs,
    validation_data=([X_vals, C_val], X_vals),
    epochs=80,
    batch_size=128,
    callbacks=cb,
    verbose=1
)

In [ ]:
X_tr_pred = model.predict([X_trs, C_tr], verbose=0)

def topk_score(x_true, x_pred, k):
    err = (x_true - x_pred) ** 2
    flat = err.reshape(len(err), -1)
    k = min(k, flat.shape[1])
    part = np.partition(flat, -k, axis=1)[:, -k:]
    return np.mean(part, axis=1)

train_scores = topk_score(X_trs, X_tr_pred, TOPK)
THRESH = float(np.percentile(train_scores, 95))

detector_meta = {"topk": int(TOPK), "threshold": THRESH}
joblib.dump(detector_meta, "../saved_model/detector_meta.joblib")

print("Detector meta saved ✅", detector_meta)

In [ ]:
model.save("../saved_model/ae_model.keras")
print("Model saved ✅")